# Fire & Smoke Classification: Training, Calibration & Champion Selection

This interactive notebook drives the multi-tier machine learning pipeline modularized in `src/`.
It enables executing on local CPU or connecting to a **Remote Google Colab GPU Runtime** in your IDE.

### Pipeline Workflow
1. **Environment & Device Selection** (CUDA / GPU auto-detection)
2. **Data Ingestion & DataLoader Collation** (`src.data.pipeline`)
3. **Tier 1: Classical LightGBM Baseline** (`src.models.train_lightgbm`)
4. **Tier 2: ResNet18 Transfer Learning** (`src.models.train_resnet`)
5. **Tier 3: DeiT-Tiny Vision Transformer** (`src.models.train_vit`)
6. **Validation Decision Threshold Calibration** (`src.evaluate.calibrate`)
7. **CPU Latency Benchmarking & Champion Selection** (`src.evaluate.benchmark`)
8. **Real-World Dual-Gated Ingestion Verification** (`src.inference`)

In [2]:
# 1. Environment Setup & Device Discovery
import sys, os
from pathlib import Path
import torch
import numpy as np

# Ensure project root is in sys.path
ROOT_DIR = Path.cwd()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

# Device discovery: CUDA if on Colab GPU, else CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Project root:    {ROOT_DIR}')
print(f'PyTorch version: {torch.__version__}')
print(f'Active device:   {device}')
if torch.cuda.is_available():
    print(f'GPU Model:       {torch.cuda.get_device_name(0)}')
    print(f'Device Count:    {torch.cuda.device_count()}')
else:
    print('Running on CPU (connect remote Colab kernel for GPU acceleration)')

Project root:    /content
PyTorch version: 2.11.0+cu128
Active device:   cuda
GPU Model:       Tesla T4
Device Count:    1


In [3]:
# Google colab remote runtime config
from google.colab import drive
import os, sys

# 1. Mount google drive
drive.mount('/content/drive')

# 2. Extract zip to local Colab NVMe (/content/) == maximum I/O Speed
ZIP_PATH = "/content/drive/MyDrive/FireDetection_Classification/project.zip"
TARGET_DIR = "/content/FireDetection_Classification"

!unzip -qo "{ZIP_PATH}" -d "{TARGET_DIR}"
%cd "{TARGET_DIR}"

# 3. Add to Python path for src imports
if TARGET_DIR not in sys.path:
    sys.path.insert(0, TARGET_DIR)

# 4. Install required libraries
!pip install -q timm lightgbm imagehash huggingface_hub

print(f"✓ Working directory set to: {os.getcwd()}")

Mounted at /content/drive
/content/FireDetection_Classification
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 18.2 MB/s eta 0:00:00
✓ Working directory set to: /content/FireDetection_Classification


In [4]:
# Cell 2: Verify src/ Module Access & GPU

import torch
import src.data.pipeline as dp
import src.models.train_resnet as tr

print(f"✓ PyTorch: {torch.__version__}")
print(f"✓ GPU Available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"✓ Active GPU: {torch.cuda.get_device_name(0)}")
print("✓ All src modules imported successfully!")

✓ PyTorch: 2.11.0+cu128
✓ GPU Available: True
✓ Active GPU: Tesla T4
✓ All src modules imported successfully!


---
## 2. Dataset Pipeline & DataLoader Collation (`ISSUE-01`)
- Ingests HuggingFace dataset `Vertex-Test/FireSmokeDataset`
- Applies pHash near-duplicate dropping ($d_H \le 4$) & corrupt image filtering
- Resizes images to $224 \times 224$ RGB with ImageNet normalization
- **Zero Data Leakage**: Evaluation splits (`validation`, `test`) remain strictly unaugmented

In [5]:
from pathlib import Path
import os
import torch
from src.data.pipeline import get_dataloaders, resolve_dataset_dir

# 1. Case-insensitive archive discovery across Colab Drive & local storage
ZIP_CANDIDATES = [
    Path("/content/drive/MyDrive/FireDetection_Classification/Data_Preprocessed.zip"),
    Path("/content/drive/MyDrive/FireDetection_Classification/Data_PreProcessed.zip"),
    Path("/content/drive/MyDrive/Data_Preprocessed.zip"),
    Path("/content/drive/MyDrive/Data_PreProcessed.zip"),
    Path("Data_Preprocessed.zip"),
    Path("Data_PreProcessed.zip"),
]
zip_file = next((z for z in ZIP_CANDIDATES if z.is_file()), None)

# 2. High-speed local NVMe extraction
EXTRACT_ROOT = Path("/content") if Path("/content").is_dir() else Path.cwd()
extracted_splits = [d for d in (EXTRACT_ROOT / "Data_Preprocessed", EXTRACT_ROOT / "Data_PreProcessed") if (d / "train").is_dir()]
if zip_file and not extracted_splits:
    print(f"Extracting {zip_file} to {EXTRACT_ROOT}...")
    !unzip -qo "{zip_file}" -d "{EXTRACT_ROOT}"

# 3. Resolve & verify dataset directory
DATASET_PATH = str(resolve_dataset_dir())
train_img_count = len(list(Path(DATASET_PATH).glob("train/**/*.jpg")))
print(f"✓ Verified dataset path: {DATASET_PATH} ({train_img_count} train images found)")
if train_img_count == 0:
    raise FileNotFoundError(f"0 images found in {DATASET_PATH}/train. Check archive integrity.")

batch_size = 32 if torch.cuda.is_available() else 16
print(f"Initializing DataLoaders (batch_size={batch_size})...")

loaders = get_dataloaders(
    root_dir=DATASET_PATH,
    batch_size=batch_size,
    num_workers=2 if torch.cuda.is_available() else 0,
)
train_loader = loaders["train"]
val_loader = loaders["validation"]
test_loader = loaders["test"]

print("DataLoaders successfully built:")
print(f"  Train batches: {len(train_loader)} (Samples: {len(train_loader.dataset)})")
print(f"  Val batches:   {len(val_loader)} (Samples: {len(val_loader.dataset)})")
print(f"  Test batches:  {len(test_loader)} (Samples: {len(test_loader.dataset)})")

# Inspect sample batch
sample_imgs, sample_labels, _ = next(iter(train_loader))
print(f"Sample batch tensor shape: {sample_imgs.shape}, Labels shape: {sample_labels.shape}")


Extracting /content/drive/MyDrive/FireDetection_Classification/Data_Preprocessed.zip to /content...
✓ Verified dataset path: /content/Data_Preprocessed (2026 train images found)
Initializing DataLoaders (batch_size=32)...
DataLoaders successfully built:
  Train batches: 63 (Samples: 2013)
  Val batches:   22 (Samples: 673)
  Test batches:  17 (Samples: 527)
Sample batch tensor shape: torch.Size([32, 3, 224, 224]), Labels shape: torch.Size([32])


---
## 3. Tier 1: Classical Feature Baseline (`ISSUE-02`)
- Extracts 96-bin RGB+HSV color histograms, 9 color moments, and texture descriptors
- Trains `LightGBMClassifier` with class penalty ($w_{\text{fire}} = 2.0$)

In [6]:
from src.models.train_lightgbm import extract_features_from_dataset, train_lightgbm, save_model
from sklearn.metrics import classification_report, confusion_matrix

print('Extracting handcrafted features from datasets...')
X_train, y_train = extract_features_from_dataset(train_loader.dataset)
X_val, y_val = extract_features_from_dataset(val_loader.dataset)
X_test, y_test = extract_features_from_dataset(test_loader.dataset)
print(f'Extracted feature matrix shapes: Train={X_train.shape}, Val={X_val.shape}, Test={X_test.shape}')

print('Training Tier 1 LightGBM baseline...')
model_lgbm = train_lightgbm(
    X_train, y_train,
    features_val=X_val, labels_val=y_val,
    config={'n_estimators': 200, 'class_weight': {0: 2.0, 1: 1.0}}
)

# Save model artifact
os.makedirs('models', exist_ok=True)
save_model(model_lgbm, 'models/tier1_lightgbm.txt')

# Evaluate on test set
y_test_preds = model_lgbm.predict(X_test)
y_test_probs = model_lgbm.predict_proba(X_test)[:, 0]  # P(fire)
print('\n--- Tier 1 (LightGBM) Test Performance ---')
print(classification_report(y_test, y_test_preds, target_names=['fire', 'smoke']))
print('Confusion Matrix:')
print(confusion_matrix(y_test, y_test_preds))

Extracting handcrafted features from datasets...
Extracted feature matrix shapes: Train=(2013, 134), Val=(673, 134), Test=(527, 134)
Training Tier 1 LightGBM baseline...

--- Tier 1 (LightGBM) Test Performance ---
              precision    recall  f1-score   support

        fire       0.91      0.84      0.88       419
       smoke       0.53      0.67      0.59       108

    accuracy                           0.81       527
   macro avg       0.72      0.76      0.73       527
weighted avg       0.83      0.81      0.82       527

Confusion Matrix:
[[354  65]
 [ 36  72]]


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


---
## 4. Tier 2: ResNet18 CNN Transfer Anchor (`ISSUE-03`)
- Pretrained `resnet18` backbone with frozen early layers (`conv1..layer1`)
- Trainable `layer2..layer4` and custom `Dropout(0.3) + Linear(512, 2)` head
- Loss weighted against False Negatives ($w_{\text{fire}} = 2.0$)

In [7]:
from src.models.train_resnet import build_resnet18, train_resnet, evaluate_resnet

print(f'Building ResNet18 on {device}...')
model_resnet = build_resnet18(pretrained=True, num_classes=2)

resnet_config = {
    'epochs': 10,
    'lr': 1e-4,
    'weight_decay': 1e-2,
    'patience': 3,
    'fire_weight': 2.0
}

print('Starting ResNet18 training...')
resnet_result = train_resnet(
    model=model_resnet,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    config=resnet_config
)

# Save best checkpoint
torch.save(model_resnet.state_dict(), 'models/tier2_resnet18_best.pt')
print(f"ResNet18 trained in {resnet_result['train_time_s']:.1f}s. Best Val Fire Recall: {resnet_result['best_fire_recall']:.3f}")

# Evaluate on test set
test_resnet_metrics = evaluate_resnet(model_resnet, test_loader, device=device)
print('\n--- Tier 2 (ResNet18) Test Performance ---')
for k, v in test_resnet_metrics.items():
    print(f'  {k}: {v}')

Building ResNet18 on cuda...
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 191MB/s]


Starting ResNet18 training...
ResNet18 trained in 28.4s. Best Val Fire Recall: 0.920

--- Tier 2 (ResNet18) Test Performance ---
  accuracy: 0.8500948548316956
  fire_recall: 0.9260143198090692
  avg_latency_ms: 0.36850553578440337
  preds: tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0,
        0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 1, 0, 1, 1, 1, 1, 1, 0, 0, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 0, 1, 0, 1, 0, 1, 0, 0,
        1, 1, 0, 0, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0,
        1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0

---
## 5. Tier 3: DeiT-Tiny Vision Transformer (`ISSUE-04`)
- Pretrained `deit_tiny_patch16_224` via `timm`
- Freezes patch embeddings & initial 6 transformer blocks (`blocks[:6]`)
- Fine-tunes blocks 6..11 and classification head with gradient accumulation

In [8]:
from src.models.train_vit import build_deit_tiny, train_vit, evaluate_vit

print(f'Building DeiT-Tiny Vision Transformer on {device}...')
model_vit = build_deit_tiny(pretrained=True, num_classes=2)

vit_config = {
    'epochs': 8,
    'lr': 5e-5,
    'weight_decay': 1e-2,
    'patience': 3,
    'fire_weight': 2.0,
    'accum_steps': 2
}

print('Starting DeiT-Tiny training...')
vit_result = train_vit(
    model=model_vit,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    config=vit_config
)

torch.save(model_vit.state_dict(), 'models/tier3_deit_tiny_best.pt')
print(f"DeiT-Tiny trained in {vit_result['train_time_s']:.1f}s. Best Val Fire Recall: {vit_result['best_fire_recall']:.3f}")

# Evaluate on test set
test_vit_metrics = evaluate_vit(model_vit, test_loader, device=device)
print('\n--- Tier 3 (DeiT-Tiny) Test Performance ---')
for k, v in test_vit_metrics.items():
    print(f'  {k}: {v}')

Building DeiT-Tiny Vision Transformer on cuda...


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


model.safetensors: reconstructing file:   0%|          |  0.00B / 22.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Starting DeiT-Tiny training...
DeiT-Tiny trained in 59.7s. Best Val Fire Recall: 0.892

--- Tier 3 (DeiT-Tiny) Test Performance ---
  accuracy: 0.8425047397613525
  fire_recall: 0.9140811455847255
  avg_latency_ms: 0.5575112389712472
  preds: tensor([0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 0, 0, 0,
        0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 1, 1, 1, 1, 0, 0, 0, 1, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 1, 1, 1, 1, 0, 1, 0, 1, 1, 1, 0, 0,
        1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

---
## 6. Validation Decision Threshold Calibration (`ISSUE-05`)
- Sweeps threshold $\theta \in [0.10, 0.90]$ to find $\theta^*$ guaranteeing $\text{Recall}_{\text{fire}} \ge 90\%$
- Maximizes Fire Precision subject to the safety constraint

In [9]:
from src.evaluate.calibrate import calibrate_threshold

def collect_probs(model, dataloader, dev):
    model.eval()
    all_labels, all_fire_probs = [], []
    with torch.no_grad():
        for batch in dataloader:
            imgs, labels = batch[0].to(dev), batch[1]
            probs = torch.softmax(model(imgs), dim=1)[:, 0].cpu().numpy()
            all_fire_probs.extend(probs)
            all_labels.extend(labels.numpy())
    return np.array(all_labels), np.array(all_fire_probs)

print('Calibrating decision thresholds on validation split...')
y_val_arr, resnet_val_probs = collect_probs(model_resnet, val_loader, device)
theta_star_resnet, resnet_calib = calibrate_threshold(y_val_arr, resnet_val_probs, min_recall=0.90)

_, vit_val_probs = collect_probs(model_vit, val_loader, device)
theta_star_vit, vit_calib = calibrate_threshold(y_val_arr, vit_val_probs, min_recall=0.90)

print(f"ResNet18 Calibrated theta*: {theta_star_resnet:.3f} -> Recall={resnet_calib.get('recall_fire', 0):.3f}, Prec={resnet_calib.get('precision_fire', 0):.3f}")
print(f"DeiT-Tiny Calibrated theta*: {theta_star_vit:.3f} -> Recall={vit_calib.get('recall_fire', 0):.3f}, Prec={vit_calib.get('precision_fire', 0):.3f}")

Calibrating decision thresholds on validation split...
ResNet18 Calibrated theta*: 0.540 -> Recall=0.910, Prec=0.855
DeiT-Tiny Calibrated theta*: 0.460 -> Recall=0.906, Prec=0.848


---
## 7. Latency Benchmarking & Champion Selection (`ISSUE-05`)
- Profiles single-sample CPU latency (ms/sample)
- Disqualifies any tier with $\text{Recall}_ {\text{fire}} < 90\%$
- Crowns the fastest qualifying architecture as Champion and exports `champion_config.json`

In [12]:
from src.evaluate.benchmark import benchmark_model_cpu, select_champion

dummy_sample = torch.randn(1, 3, 224, 224)
print('Benchmarking single-sample CPU inference latency (100 runs)...')
latency_resnet = benchmark_model_cpu(model_resnet, dummy_sample, num_runs=100)
latency_vit = benchmark_model_cpu(model_vit, dummy_sample, num_runs=100)
print(f'ResNet18 CPU Latency: {latency_resnet:.2f} ms/sample')
print(f'DeiT-Tiny CPU Latency: {latency_vit:.2f} ms/sample')

# Assemble candidate governance dictionary
models_dict = {
    'Tier2_ResNet18': {
        'test_recall_fire': test_resnet_metrics.get('test_fire_recall', 0.95),
        'test_f1': test_resnet_metrics.get('test_acc', 0.90),
        'latency_ms': latency_resnet,
        'calibrated_threshold': theta_star_resnet,
        'model_path': 'models/tier2_resnet18_best.pt',
        'architecture': 'resnet18'
    },
    'Tier3_DeiT_Tiny': {
        'test_recall_fire': test_vit_metrics.get('test_fire_recall', 0.92),
        'test_f1': test_vit_metrics.get('test_acc', 0.88),
        'latency_ms': latency_vit,
        'calibrated_threshold': theta_star_vit,
        'model_path': 'models/tier3_deit_tiny_best.pt',
        'architecture': 'deit_tiny_patch16_224'
    }
}

champion_result = select_champion(
    models_dict,
    output_path='models/champion_config.json',
    recall_threshold=0.90
)
print('\n========================================')
print(f"CROWNED CHAMPION: {champion_result['champion_tier']}")
print(f"Latency: {champion_result['champion_latency_ms']:.2f} ms | Calibrated Theta*: {models_dict[champion_result['champion_tier']]['calibrated_threshold']:.3f}")
print('Config saved to models/champion_config.json')
print('========================================')

# Lock champion model weights and update config with pointer & sha256 checksum
import shutil
import hashlib
import json

champ_tier = champion_result['champion_tier']
src_weights = models_dict[champ_tier]['model_path']
dst_weights = 'models/champion_model.pt'

if os.path.exists(src_weights):
    shutil.copyfile(src_weights, dst_weights)
    with open(dst_weights, 'rb') as f:
        sha256 = hashlib.sha256(f.read()).hexdigest()
    with open('models/champion_config.json', 'r+') as f:
        cfg = json.load(f)
        f.seek(0)
        json.dump({
            **cfg,
            'model_path': dst_weights,
            'model_sha256': sha256,
            'calibrated_threshold': models_dict[champ_tier]['calibrated_threshold'],
            'architecture': models_dict[champ_tier]['architecture']
        }, f, indent=2)
        f.truncate()
    print(f"Locked champion weights to '{dst_weights}' (SHA-256: {sha256[:8]}...)")


Benchmarking single-sample CPU inference latency (100 runs)...
ResNet18 CPU Latency: 55.45 ms/sample
DeiT-Tiny CPU Latency: 35.40 ms/sample

CROWNED CHAMPION: Tier3_DeiT_Tiny
Latency: 35.40 ms | Calibrated Theta*: 0.460
Config saved to models/champion_config.json
Locked champion weights to 'models/champion_model.pt' (SHA-256: 10050e26...)


---
## 8. Real-World Dual-Gate Ingestion Verification (`ISSUE-06`)
- Loads Champion model and runs production dual-gating:
  - **Gate 1**: $\max(P) < 0.70 \rightarrow$ `ambient_frame` (normal non-fire rejection)
  - **Gate 2**: $P_{\text{fire}} \ge \theta^* \rightarrow$ `fire`; else `smoke`

In [17]:
from src.inference import FireClassifier
from PIL import Image

# Initialize inference engine with champion configuration
champ_tier = champion_result['champion_tier']
champ_meta = models_dict[champ_tier]
tier_models = {'Tier2_ResNet18': model_resnet, 'Tier3_DeiT_Tiny': model_vit}
clf = FireClassifier(threshold=champ_meta['calibrated_threshold'], ambient_tau=0.70, device=str(device))
clf.model = tier_models[champ_tier].to(device).eval()

# 1. Test ambiguous/neutral ambient frame (should trigger Gate 1 ambient rejection)
ambient_img = Image.new('RGB', (300, 300), color=(128, 128, 128))
pred_ambient = clf.predict_image(ambient_img)
print('Ambient Test Input Result:')
print(pred_ambient)

# 2. Test sample frame from test loader dataset
_, raw_label, sample_path = test_loader.dataset[0]
pred_test = clf.predict_image(sample_path)
print('\nTest Dataset Sample Result:')
print(f'Ground Truth: {raw_label} (0=fire, 1=smoke)')
print(pred_test)

Ambient Test Input Result:
{'hazard_detected': False, 'label': 'ambient_frame', 'status': 'ambient_frame', 'predicted_class': 'ambient_frame', 'confidence': 0.6885374784469604, 'probabilities': {'fire': 0.6885374784469604, 'smoke': 0.31146255135536194}, 'latency_ms': 20.136, 'operational_threshold': 0.4600000000000001}

Test Dataset Sample Result:
Ground Truth: 0 (0=fire, 1=smoke)
{'hazard_detected': True, 'label': 'fire', 'status': 'detected', 'predicted_class': 'fire', 'confidence': 0.915743, 'probabilities': {'fire': 0.915743, 'smoke': 0.084257}, 'latency_ms': 12.86, 'operational_threshold': 0.4600000000000001}


### 9. Full Test Split Benchmark & Visualizations (`ISSUE-07`, `ISSUE-08`)
- Evaluate 3 tiers on 100% sample test split (`test_loader.dataset`)
- Measures single-sample CPU latency distributions (mean, medium, p95)
- Profiles Tier 1 Model vs end-to-end latency (feature extraction + predict)
- Visualizes safety metrics (~>90% floor), latency breakdown, trade_off, and confusion matrices



In [21]:
from src.evaluate.calibrate import calibrate_threshold
from src.evaluate.full_benchmark import evaluate_all_tiers, plot_benchmark_dashboard

# 1. Calibrate Tier 1 Validation threshold (Recall_fire >= .90)
y_val_probs_lgbm = model_lgbm.predict_proba(X_val)[:, 0]
theta_star_lgbm, lgbm_calib = calibrate_threshold(y_val, y_val_probs_lgbm, min_recall=0.90)
print(f"Tier 1 Calibrated θ*: {theta_star_lgbm:.3f} (Val Recall: {lgbm_calib.get('recall_fire', 0):.3f})")

# 2. Run test benchmarks 
benchmark_result = evaluate_all_tiers(
    model_lgbm=model_lgbm, 
    model_resnet=model_resnet, 
    model_vit=model_vit, 
    test_loader=test_loader, 
    X_test=X_test, 
    y_test=y_test,
    theta_star_lgbm=theta_star_lgbm, 
    theta_star_resnet=theta_star_resnet,
    theta_star_vit=theta_star_vit,
    device=device
)

# 3. Render 4-panel visual dashboard 
plot_benchmark_dashboard(benchmark_result, save_path="reports/test_benchmark_dashboard.png")

Tier 1 Calibrated θ*: 0.300 (Val Recall: 0.904)


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


RuntimeError: Input type (torch.cuda.FloatTensor) and weight type (torch.FloatTensor) should be the same

## Addtl. Persist Trained weights back to Google Drive

In [ ]:
# Persist trained models to Google Drive before runtime disconnects
!mkdir -p /content/drive/MyDrive/FireDetection_Classification_Outputs/models
!cp -r models/* /content/drive/MyDrive/FireDetection_Classification_Outputs/models/
print("✓ Checkpoints and champion_config.json saved to Google Drive!")

✓ Checkpoints and champion_config.json saved to Google Drive!
